In [1]:
import pandas as pd

df = pd.read_csv('rolex.v1.txt', sep='\t', header=None, names=['word_form', 'lemma', 'MSD_tag', 'syllabification', 'lexical_stress', 'phonetic_transcription'])

df.head(50)

,word_form,lemma,MSD_tag,syllabification,lexical_stress,phonetic_transcription
0,Ăstea,acesta,Dd3fpr---o,ăs.tea,'ăstea,@ s t e_X a
1,ă,=,I,ă,ă,@
2,ăi,acel,Dd3mpr---e,ăi,ăi,@ j
3,ăi,acela,Pd3mpr,ăi,ăi,@ j
4,ăi,vrea,Va--2s,ăi,ăi,@ j
5,ăia,acela,Dd3mpr---o,ă.ia,'ăia,@ i a
6,ăia,acela,Pd3mpr,ă.ia,'ăia,@ i a
7,ăl,acel,Dd3msr---e,ăl,ăl,@ l
8,ăl,cel,Tdmsr,ăl,ăl,@ l
9,ăla,acela,Dd3msr---o,ă.la,'ăla,@ l a


In [2]:
metadata = pd.read_csv('../ro_vsr/metadata_ready.csv')
duration_df = pd.read_csv('../ro_vsr/metadata_clean.csv', usecols=['file_path', 'duration'])
duration_df = duration_df.drop_duplicates(subset='file_path', keep='first')
metadata = metadata.merge(duration_df, left_on='clip_path', right_on='file_path', how='left')
metadata = metadata.drop(columns=['file_path'])


In [3]:
import os

def lookup_token(token, word_form_index):
    # Try exact match
    if token in word_form_index:
        return word_form_index[token]
    # Try lowercasing the token
    if token.lower() in word_form_index:
        return word_form_index[token.lower()]
    # Try capitalizing first char of token
    capitalized = token[0].upper() + token[1:] if token else token
    if capitalized in word_form_index:
        return word_form_index[capitalized]
    return None

def lookup_token_with_hyphen(token, word_form_index):
    # First try direct lookup (all casing variants)
    result = lookup_token(token, word_form_index)
    if result is not None:
        return result

    # If hyphenated, split and look up each part
    if '-' in token:
        parts = token.split('-')
        phoneme_parts = []
        for part in parts:
            if not part:  # skip empty from leading/trailing hyphens
                continue
            p = lookup_token(part, word_form_index)
            if p is None:
                return None  # any part missing -> whole token fails
            phoneme_parts.append(p)
        if phoneme_parts:
            return ' '.join(phoneme_parts)
    return None

# Build phoneme index from rolex (keep='last' to prefer common POS entries)
word_form_index = df.drop_duplicates('word_form', keep='last').set_index('word_form')['phonetic_transcription'].to_dict()

# Load manual entries for frequent missing words
extra = pd.read_csv('frequent_missing.tsv', sep='\t', header=None,
                     names=['word_form', 'lemma', 'MSD_tag', 'syllabification', 'lexical_stress', 'phonetic_transcription'])
word_form_index.update(extra.drop_duplicates('word_form').set_index('word_form')['phonetic_transcription'].to_dict())

# Load LLM-generated entries (from fill_unk.ipynb)
llm_file = 'llm_generated_missing.tsv'
n_llm = 0
if os.path.exists(llm_file):
    llm_extra = pd.read_csv(llm_file, sep='\t', header=None,
                             names=['word_form', 'lemma', 'MSD_tag', 'syllabification', 'lexical_stress', 'phonetic_transcription'])
    word_form_index.update(llm_extra.drop_duplicates('word_form').set_index('word_form')['phonetic_transcription'].to_dict())
    n_llm = len(llm_extra)

print(f"Phoneme index: {len(word_form_index)} entries (rolex + {len(extra)} manual + {n_llm} LLM)")
print(f"  și -> {word_form_index.get('și')}")
print(f"  e  -> {word_form_index.get('e')}")


Phoneme index: 284277 entries (rolex + 15 manual + 11953 LLM)
  și -> S i
  e  -> je


In [4]:
from collections import Counter

UNK = 'UNK'

def phonemize_sentence(transcript, word_form_index):
    tokens = transcript.split()
    phoneme_list = []
    n_missing = 0
    max_consecutive = 0
    current_consecutive = 0
    for token in tokens:
        phonemes = lookup_token_with_hyphen(token, word_form_index)
        if phonemes is not None:
            phoneme_list.append(phonemes)
            current_consecutive = 0
        else:
            phoneme_list.append(UNK)
            n_missing += 1
            current_consecutive += 1
            max_consecutive = max(max_consecutive, current_consecutive)
    return phoneme_list, n_missing, max_consecutive, len(tokens)

# Process all sentences
results = []
for _, row in metadata.iterrows():
    transcript = row['transcript']
    if pd.isna(transcript):
        continue
    phoneme_list, n_missing, max_consecutive, n_tokens = phonemize_sentence(transcript, word_form_index)
    missing_ratio = n_missing / n_tokens if n_tokens else 0
    results.append({
        'clip_path': row['clip_path'],
        'transcript': transcript,
        'phonemes': ' '.join(phoneme_list),
        'n_tokens': n_tokens,
        'n_missing': n_missing,
        'missing_ratio': missing_ratio,
        'max_consecutive_missing': max_consecutive,
        'split': row.get('split', ''),
        'duration': row.get('duration', float('nan')),
    })

results_df = pd.DataFrame(results)

# Drop: > 1/3 missing tokens OR >= 2 consecutive missing
drop_high_missing = results_df['missing_ratio'] > 1/3
drop_consecutive = results_df['max_consecutive_missing'] >= 2
drop_mask = drop_high_missing | drop_consecutive
clean_df = results_df[~drop_mask].copy()

# Coverage stats
token_counts = Counter(token for t in metadata['transcript'].dropna() for token in t.split())
total_occ = sum(token_counts.values())
matched_occ = sum(c for t, c in token_counts.items() if lookup_token_with_hyphen(t, word_form_index) is not None)

print(f"Absolute token coverage: {matched_occ/total_occ:.2%}")
print(f"Total sentences: {len(results_df)}")
print(f"Dropped (>33% missing): {drop_high_missing.sum()} ({drop_high_missing.mean():.2%})")
print(f"Dropped (≥2 consecutive missing): {drop_consecutive.sum()} ({drop_consecutive.mean():.2%})")
print(f"Dropped (either): {drop_mask.sum()} ({drop_mask.mean():.2%})")
print(f"Kept: {len(clean_df)} ({(~drop_mask).mean():.2%})")
print(f"  Fully phonemized: {(clean_df['n_missing'] == 0).sum()}")
print(f"  With UNK: {(clean_df['n_missing'] > 0).sum()}")


Absolute token coverage: 99.98%
Total sentences: 39929
Dropped (>33% missing): 1 (0.00%)
Dropped (≥2 consecutive missing): 8 (0.02%)
Dropped (either): 8 (0.02%)
Kept: 39921 (99.98%)
  Fully phonemized: 39773
  With UNK: 148


In [5]:
results_df = results_df[~results_df['phonemes'].str.contains(r'\bUNK\b', regex=True, case=True)]
len(results_df)

39773

In [6]:
from pathlib import Path

clips_dir = Path('../clips')

existing_clips = {path.stem for path in clips_dir.rglob('*.avi')}

results_df = results_df[results_df['clip_path'].isin(existing_clips)]

len(results_df)

39660

In [7]:
results_df = results_df[results_df['duration'] <= 30].copy()
fin_df = results_df[['clip_path', 'transcript', 'phonemes', 'split']]
fin_df.to_csv('../labels.csv', index=False)

split_lengths = results_df.groupby('split')['duration'].sum().sort_values(ascending=False)

print('Split lengths after all filtering:')
for split_name, total_seconds in split_lengths.items():
    print(f"{split_name}: {total_seconds / 3600:.2f}h ({total_seconds / 60:.2f}m)")

fin_df.head(50)

Split lengths after all filtering:
train: 100.45h (6026.94m)
val: 11.25h (674.89m)
test: 10.33h (619.50m)


,clip_path,transcript,phonemes,split
0,andreea_antonescu_qahn56pteSA_clip_238,că au avut ceva de învățat de aici și sigur da...,k @ a w a v u t tS e v a d e 1 n v @ ts a t d ...,train
1,bianca_nutu_4QMZpa_zYKU_clip_81,ceea ce gândești ceea ce simți despre ceea ce ...,tS e e_X a tS e g 1 n d e S t i_0 tS e e_X a t...,train
2,iulia_parlea_AMpklgShYD0_clip_279,maică-mea avea o chestie cu mințitul deci ea d...,m a j k @ m e_X a a v e_X a o k_j e s t i e k ...,train
3,ana_morodan_aDOzKnPltgw_clip_210,tinerilor care au probleme din zona digitală,t i n e r i l o r k a r e a w p r o b l e m e ...,train
4,lolrelai_sCOjyP2ZdJo_clip_400,doi la mână eu am aflat că nu s-au luat camere...,d o j l a m 1 n @ e w a m a f l a t k @ n u s ...,train
5,irina_petrea_CO4P03D_Hl0_clip_190,dar fără să fiu foarte insistent dacă refuză s...,d a r f @ r @ s @ f i w f o_X a r t e i n s i ...,train
6,ioana_ginghina_WC5SA66nEIY_clip_68,cei doi clooney și cu brad pitt frumoși așa ș...,tS e j d o j k l u n i S i k u b r a d p i t f...,train
7,mihaela_tatu_kD4lJehL23Q_clip_291,nu nu seara înainte de culcare îi mulțumesc pe...,n u n u s e_X a r a 1 n a j n t e d e k u l k ...,train
8,adrian_alexandrov_G1h5PBrtzds_clip_85,nu pe mine personal nu toți prietenii mei sunt...,n u p e m i n e p e r s o n a l n u t o ts i_0...,train
9,catalin_bordea_zlRnts0-FWg_clip_179,mă știu de foarte mult timp cu ei dar acum noi...,m @ S t i u d e f o_X a r t e m u l t t i m p ...,train
